**Importazione e configurazioni varie**

In [ ]:
from google.colab import drive
import os

# Collega il drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Installazione librerie in corso...")

#YOLO e Supervision
!pip install -q ultralytics supervision

print("Installazione completata con successo!")

Installazione librerie in corso... (richiederà un paio di minuti)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 23.0 MB/s eta 0:00:00
Installazione completata con successo!


In [ ]:
!pip install git+https://github.com/facebookresearch/sam3.git

  Cloning https://github.com/facebookresearch/sam3.git to /tmp/pip-req-build-zhbep4pw
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam3.git /tmp/pip-req-build-zhbep4pw
  Resolved https://github.com/facebookresearch/sam3.git to commit 44ef224799e7b32e017855823883909d38ffb30f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 135.5 MB/s eta 0:00:00
  Created wheel for sam3: filename=sam3-0.1.0-py3-none-any.whl size=2032355 sha256=7f01d5edc234bcdc86f2d8cd6aad18ca7a331ceffe557640603db3de7d0f4457
  Stored in directory: /tmp/pip-ephem

------------------------------------------------------------------

**Pipeline: YOLOv11 + SAM3 (su singolo frame)**

**YOLOv11 + SAM3 (su singolo frame) con fix (v2)**

In [ ]:
"""
Pipeline v8 — SAM3 output GREZZO (nessuna levigatura) con pesi sbilanciati su SAM
==============================================================================
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from PIL import Image

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# ─────────────────────────────────────────────────────────────────────────────
# Configurazione
# ─────────────────────────────────────────────────────────────────────────────

YOLO_MODEL_PATH         = "/content/drive/MyDrive/ProgettoDL/risultati_yolov11nseg/esperimento_seg11_datasetunificato/weights/best.pt"
IMAGE_PATH              = "/content/frametest30.png"
SAM3_MAX_SIDE           = 768
YOLO_BOX_CONF_THRESHOLD = 0.4

# Soglia minima combinata abbassata perché la maschera di YOLO è scadente
SAM3_MIN_COMBINED_THRESHOLD = 0.05

# PESI INVERTITI: Ci fidiamo all'80% di SAM3 e solo al 20% della forma di YOLO
W_IOU   = 0.2
W_SCORE = 0.8

# ─────────────────────────────────────────────────────────────────────────────
# Lista completa dei prompt candidati
# ─────────────────────────────────────────────────────────────────────────────

STADIUM_AD_PROMPTS = [
    # ── Termini generici base ─────────────────────────────────────────────────
    "advertising board",
    "stadium ad hoarding",
    "sponsor billboard",
    "commercial advertising panel",
    "pitch-side advertising panel",
    "advertising hoarding",
    "sponsor panel",
    "advertisement board",
    "ad board",
    "sponsor board",

    # ── NUOVO: Profondità e Sfondo (Focus sul pannello posteriore) ────────────
    "background advertising board",
    "rear sponsor hoarding",
    "advertising panel in the far background",
    "physical stadium wall behind the pitch",
    "sponsor boards behind the players",
    "backdrop advertising fence",
    "far end perimeter hoarding",
    "stadium wall behind the goal net",
    "structural advertising wall in the background",
    "physical perimeter board at the back",

    # ── NUOVO: Colore e Contrasto (Specifici per forzare la separazione) ──────
    "red advertising board",
    "bright red sponsor panel",
    "red perimeter hoarding with white logos",
    "red LED board behind the goal",
    "sponsor signs with red background",
    "solid red wall with advertisements",
    "stadium background wall painted red",
    "purple virtual advertising board",
    "semi-transparent purple sponsor banner",
    "orange and yellow advertising board",
    "yellow physical stadium wall behind the pitch",
    "red advertising board",

    # ── Posizione perimetrale ─────────────────────────────────────────────────
    "perimeter sponsor hoarding along the field",
    "sideline advertising boards",
    "advertising banners behind the goal",
    "stadium boundary wall with advertisements",
    "field perimeter sponsor signs",
    "pitch perimeter advertising board",
    "touchline advertising hoarding",
    "goal line advertising board",
    "byline sponsor board",
    "behind the goal sponsor hoarding",
    "corner flag advertising panel",
    "pitch boundary advertising fence",
    "along the pitch advertising wall",
    "advertising board behind the goal net",
    "stadium wall right behind the goal",

    # ── Forma e geometria ─────────────────────────────────────────────────────
    "long rectangular advertising banner",
    "continuous flat boundary ad board",
    "long horizontal sponsor signs",
    "low rectangular perimeter wall",
    "flat horizontal rectangular panel with logo",
    "wide low banner at pitch level",
    "thin horizontal strip with sponsor name",
    "elongated rectangular sign along the grass",
    "narrow horizontal hoarding at ground level",
    "low flat panel running along the pitch",

    # ── Tecnologia LED / digitale ─────────────────────────────────────────────
    "bright LED advertising screen",
    "digital LED perimeter board",
    "electronic glowing billboard",
    "scrolling digital ad display",
    "animated LED sponsor board",
    "electronic perimeter display screen",
    "digital rotating advertisement panel",
    "glowing electronic sideline screen",
    "LED ribbon board along the pitch",
    "virtual advertising board overlay",
    "high brightness digital signage panel",
    "backlit sponsor display board",

    # ── Stampa statica / tradizionale ─────────────────────────────────────────
    "static printed advertising banner",
    "non-digital flat advertising board",
    "solid color sponsor panel with text",
    "white board with sponsor logo",
    "colored rectangular printed advertisement",
    "painted physical sponsor wall",

    # ── Contenuto / brand ─────────────────────────────────────────────────────
    "panels displaying brand logos",
    "commercial sponsor graphics along the pitch",
    "colorful advertising text and logos on boards",
    "brand advertisement along the touchline",
    "company name on stadium hoarding",
    "branding panel next to the playing field",
    "sports brand logo on advertising board",
    "asroma on advertising board"

    # ── Contesto stadio calcio ────────────────────────────────────────────────
    "football stadium perimeter advertising",
    "premier league advertising hoarding",
    "champions league pitch-side sponsor panel",
    "football ground commercial display",
    "professional football match ad board",
    "live football game sponsor signage",
    "match day advertising hoarding in stadium",
    "stadium inner perimeter commercial panel",
    "football club sponsor display fence",
    "rectangular sign at the edge of football pitch",
    "advertising strip running along football field",
    "low wall with sponsor branding at stadium",
]

# ─────────────────────────────────────────────────────────────────────────────
# Utility generali
# ─────────────────────────────────────────────────────────────────────────────

def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    intersection = (a & b).sum()
    union        = (a | b).sum()
    return float(intersection / union) if union > 0 else 0.0

def bbox_iou(mask: np.ndarray, box_xyxy: np.ndarray) -> float:
    x1, y1, x2, y2 = box_xyxy.astype(int)
    box_mask = np.zeros_like(mask, dtype=bool)
    box_mask[y1:y2, x1:x2] = True
    return mask_iou(mask, box_mask)

def clip_mask_to_bbox(mask: np.ndarray, box_xyxy: np.ndarray) -> np.ndarray:
    x1, y1, x2, y2 = box_xyxy.astype(int)
    clipped = np.zeros_like(mask, dtype=bool)
    # Applichiamo un leggero margine (+/- 15 pixel) per evitare che un bbox YOLO
    # leggermente impreciso tagli via bordi buoni della maschera di SAM3
    H, W = mask.shape
    y1_m, y2_m = max(0, y1-15), min(H, y2+15)
    x1_m, x2_m = max(0, x1-15), min(W, x2+15)
    clipped[y1_m:y2_m, x1_m:x2_m] = mask[y1_m:y2_m, x1_m:x2_m]
    return clipped

def mask_centroid(mask: np.ndarray):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return float(xs.mean()), float(ys.mean())

def resize_image(image_bgr: np.ndarray, max_side: int = 768):
    H, W      = image_bgr.shape[:2]
    long_side = max(H, W)
    if long_side <= max_side:
        return image_bgr, 1.0
    scale   = max_side / long_side
    resized = cv2.resize(image_bgr,
                         (int(W * scale), int(H * scale)),
                         interpolation=cv2.INTER_AREA)
    return resized, scale

def upscale_mask(mask: np.ndarray, target_H: int, target_W: int) -> np.ndarray:
    up = cv2.resize(mask.astype(np.uint8) * 255,
                    (target_W, target_H), interpolation=cv2.INTER_NEAREST)
    return up > 127

def annotate_masks_manual(scene: np.ndarray, detections: sv.Detections, opacity: float = 0.6) -> np.ndarray:
    COLORS = [(255,80,80),(80,255,80),(80,80,255),(255,255,80),(255,80,255),(80,255,255)]
    output = scene.copy()
    for i, mask in enumerate(detections.mask):
        if mask.sum() == 0: continue
        color   = COLORS[i % len(COLORS)]
        overlay = scene.copy()
        overlay[mask] = color
        output  = cv2.addWeighted(overlay, opacity, output, 1 - opacity, 0)
        del overlay
        # Disegna contorno sottile per visualizzazione
        m8 = mask.astype(np.uint8) * 255
        cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(output, cnts, -1, color, thickness=2)
        del m8, cnts
    gc.collect()
    return output

def show_titled(image_bgr: np.ndarray, title: str):
    if image_bgr.ndim == 2:
        image_bgr = cv2.cvtColor(image_bgr, cv2.COLOR_GRAY2BGR)
    banner = np.zeros((40, image_bgr.shape[1], 3), dtype=np.uint8)
    cv2.putText(banner, title, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    cv2_imshow(np.vstack([banner, image_bgr]))
    del banner
    gc.collect()

def vram_mb() -> str:
    u = torch.cuda.memory_allocated() / 1024**2
    r = torch.cuda.memory_reserved()  / 1024**2
    return f"VRAM alloc={u:.0f}MB resv={r:.0f}MB"

def free_vram():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def deep_unload(obj) -> None:
    if obj is None: return
    try: obj.cpu()
    except Exception: pass
    cpu_empty = torch.empty(0, device="cpu")
    try:
        for p in obj.parameters():
            if p.data.is_cuda: p.data = cpu_empty
            if p.grad is not None: p.grad = None
    except Exception: pass
    try:
        for b in obj.buffers():
            if b.is_cuda: b.data = cpu_empty
    except Exception: pass
    try:
        for child in obj.children(): deep_unload(child)
    except Exception: pass
    try:
        obj._forward_hooks.clear()
        obj._backward_hooks.clear()
        obj._forward_pre_hooks.clear()
    except Exception: pass

# ─────────────────────────────────────────────────────────────────────────────
# Wrapper inferenza SAM3
# ─────────────────────────────────────────────────────────────────────────────

# IMPOSTATI A FALSE: Sappiamo che set_text_prompt non vuole box e punti.
_SAM3_SUPPORTS_BOXES  = False
_SAM3_SUPPORTS_POINTS = False

def call_sam3(processor, state, prompt: str, box_xyxy_sam: np.ndarray | None, point_xy_sam: tuple | None, device: str) -> dict | None:
    global _SAM3_SUPPORTS_BOXES, _SAM3_SUPPORTS_POINTS
    kwargs: dict = {"state": state, "prompt": prompt}

    if box_xyxy_sam is not None and _SAM3_SUPPORTS_BOXES:
        kwargs["boxes"] = torch.tensor(box_xyxy_sam, dtype=torch.float32, device=device).unsqueeze(0)

    if point_xy_sam is not None and _SAM3_SUPPORTS_POINTS:
        kwargs["points"] = torch.tensor([[point_xy_sam]], dtype=torch.float32, device=device)
        kwargs["point_labels"] = torch.ones((1, 1), dtype=torch.int64, device=device)

    try:
        with torch.inference_mode():
            with torch.autocast(device, dtype=torch.bfloat16):
                output = processor.set_text_prompt(**kwargs)
        if output["masks"].shape[0] == 0:
            return None
        return output
    except Exception as e:
        print(f"  [API] Errore SAM3: {e}")
        return None

# ─────────────────────────────────────────────────────────────────────────────
# Selezione dinamica del prompt per un singolo pannello
# ─────────────────────────────────────────────────────────────────────────────

def select_best_prompt_for_panel(
    processor, state, device: str, box_sam: np.ndarray, point_sam: tuple | None,
    yolo_mask_orig: np.ndarray | None, box_orig: np.ndarray, H_orig: int, W_orig: int, prompts: list[str],
) -> tuple[str, float, np.ndarray, float]:

    best_prompt   = prompts[0]
    best_combined = -1.0
    best_mask_sam = None
    best_score    = 0.0

    for prompt in prompts:
        output = call_sam3(processor, state, prompt, box_sam, point_sam, device)
        if output is None:
            continue

        n_out      = output["masks"].shape[0]
        masks_out  = output["masks"][:, 0].cpu().numpy().astype(bool)
        scores_out = output["scores"].cpu().float().numpy()
        del output
        free_vram()

        best_k_local = 0
        best_iou_local = -1.0
        for k in range(n_out):
            m_up  = upscale_mask(masks_out[k], H_orig, W_orig)
            iou_k = (mask_iou(m_up, yolo_mask_orig) if yolo_mask_orig is not None else bbox_iou(m_up, box_orig))
            if iou_k > best_iou_local:
                best_iou_local = iou_k
                best_k_local   = k
            del m_up

        iou_sel  = best_iou_local
        sc_sel   = float(scores_out[best_k_local])
        combined = iou_sel * W_IOU + sc_sel * W_SCORE

        print(f"      iou={iou_sel:.3f} score={sc_sel:.3f} combined={combined:.3f} | '{prompt}'")

        if combined > best_combined:
            best_combined = combined
            best_prompt   = prompt
            best_mask_sam = masks_out[best_k_local].copy()
            best_score    = sc_sel

        del masks_out, scores_out
        gc.collect()

    return best_prompt, best_combined, best_mask_sam, best_score

# ─────────────────────────────────────────────────────────────────────────────
# Pipeline principale
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(yolo_model_path: str = YOLO_MODEL_PATH, image_path: str = IMAGE_PATH, sam3_max_side: int = SAM3_MAX_SIDE):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    device = "cuda" if torch.cuda.is_available() else "cpu"

    free_vram()
    yolo_model, sam_model, processor, shared_state = None, None, None, None

    try:
        image_bgr = cv2.imread(image_path)
        H_orig, W_orig = image_bgr.shape[:2]
        image_rgb_full = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        image_bgr_sam, scale = resize_image(image_bgr, max_side=sam3_max_side)
        H_sam, W_sam = image_bgr_sam.shape[:2]
        image_pil_sam = Image.fromarray(cv2.cvtColor(image_bgr_sam, cv2.COLOR_BGR2RGB))
        del image_bgr_sam
        gc.collect()

        print("\n[STEP 1] Rilevamento YOLO...")
        yolo_model = YOLO(yolo_model_path)
        with torch.no_grad():
            yolo_results = yolo_model(image_rgb_full, conf=0.1, iou=0.3, device="cpu")[0]
        yolo_det = sv.Detections.from_ultralytics(yolo_results)
        del image_rgb_full, yolo_results, yolo_model
        free_vram()

        n_yolo = len(yolo_det)
        if n_yolo == 0: return

        vis_yolo = image_bgr.copy()
        if yolo_det.mask is not None: vis_yolo = annotate_masks_manual(vis_yolo, yolo_det, opacity=0.4)
        vis_yolo = sv.BoxAnnotator(thickness=2).annotate(scene=vis_yolo, detections=yolo_det)
        show_titled(vis_yolo, "STEP 1 YOLO: bbox + maschera iniziale")
        del vis_yolo

        print("\n[STEP 2] Caricamento SAM3 + embedding...")
        sam_model = build_sam3_image_model()
        processor = Sam3Processor(sam_model, confidence_threshold=0.3)
        free_vram()

        with torch.inference_mode():
            with torch.autocast(device, dtype=torch.bfloat16):
                shared_state = processor.set_image(image_pil_sam)
        del image_pil_sam
        free_vram()

        print("\n[STEP 3] Selezione dinamica del prompt...")
        final_masks, final_scores, final_prompts, fallback_used = [], [], [], []

        for idx in range(n_yolo):
            box_orig = yolo_det.xyxy[idx]
            box_sam  = box_orig * scale
            yolo_mask_orig = yolo_det.mask[idx] if yolo_det.mask is not None else None
            point_sam = mask_centroid(upscale_mask(yolo_mask_orig, H_sam, W_sam)) if yolo_mask_orig is not None else (float(box_sam[0]+box_sam[2])/2, float(box_sam[1]+box_sam[3])/2)

            best_prompt, best_combined, best_mask_sam, best_score = select_best_prompt_for_panel(
                processor, shared_state, device, box_sam, point_sam, yolo_mask_orig, box_orig, H_orig, W_orig, STADIUM_AD_PROMPTS
            )

            print(f"    ▶ Scelto: '{best_prompt}' combined={best_combined:.3f} score={best_score:.3f}")

            if best_combined < SAM3_MIN_COMBINED_THRESHOLD or best_mask_sam is None:
                # FALLBACK YOLO (Nessuna maschera buona da SAM)
                if yolo_mask_orig is not None: m_fb = clip_mask_to_bbox(yolo_mask_orig, box_orig)
                else:
                    m_fb = np.zeros((H_orig, W_orig), dtype=bool)
                    m_fb[int(box_orig[1]):int(box_orig[3]), int(box_orig[0]):int(box_orig[2])] = True
                final_masks.append(m_fb)
                final_scores.append(float(yolo_det.confidence[idx]))
                final_prompts.append("(fallback YOLO)")
                fallback_used.append(True)
            else:
                # SUCCESSO SAM3: MASCHERA GREZZA (Nessun cv2.approxPolyDP, Nessun Morph)
                m_orig = upscale_mask(best_mask_sam, H_orig, W_orig)
                m_clipped = clip_mask_to_bbox(m_orig, box_orig)
                final_masks.append(m_clipped)
                final_scores.append(best_score)
                final_prompts.append(best_prompt)
                fallback_used.append(False)

        del shared_state; deep_unload(processor); deep_unload(sam_model)
        free_vram()

        print("\n[STEP 4] Visualizzazione...")
        valid_idx, boxes_list = [], []
        for i, m in enumerate(final_masks):
            ys, xs = np.where(m)
            if len(xs) > 0:
                boxes_list.append(np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=float))
                valid_idx.append(i)

        if not valid_idx: return

        masks_np  = np.stack([final_masks[i] for i in valid_idx])
        scores_np = np.array([final_scores[i] for i in valid_idx], dtype=np.float32)
        boxes_np  = np.stack(boxes_list)
        fallback_f= [fallback_used[i] for i in valid_idx]

        sam_det = sv.Detections(xyxy=boxes_np, mask=masks_np, confidence=scores_np, class_id=np.zeros(len(boxes_np), dtype=int))

        comp_bw = np.zeros((H_orig, W_orig), dtype=np.uint8)
        for m in masks_np: comp_bw[m] = 255
        show_titled(cv2.cvtColor(comp_bw, cv2.COLOR_GRAY2BGR), "STEP 4 Maschera SAM3 GREZZA")

        labels = [f"[{'YOLO' if fallback_f[i] else 'SAM3'}] {scores_np[i]:.2f}" for i in range(len(masks_np))]
        annotated = annotate_masks_manual(image_bgr.copy(), sam_det, opacity=0.4)
        annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=sam_det)
        annotated = sv.LabelAnnotator(text_scale=0.5).annotate(scene=annotated, detections=sam_det, labels=labels)
        show_titled(annotated, "STEP 4 Risultato finale")

    finally:
        free_vram()

run_pipeline()

Output hidden; open in https://colab.research.google.com to view.

----------------------------------------------------------------------------------------

**Pipeline: YOLOv11n-seg (SAHI) + SAM3 (su video)**

In [ ]:
!pip install sahi -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.4 MB/s eta 0:00:00


In [ ]:
"""
Pipeline Video con SAM3 in modalità VIDEO —

Miglioramenti rispetto alla v1:
  [fix #1] SAHI tiling su YOLO → cattura pannelli piccoli/lontani
  [fix #2] Soglie di confidenza adattive in base all'area del box
  [fix #3] Crop + upscale del ROI prima della selezione prompt SAM3
  [fix #4] is_adboard_shape adattivo: soglie più permissive per box piccoli
  [fix #5] Anchor frame smart: si sceglie il frame con più detection nella finestra

Architettura invariata:
  - SAM3 video mode gestisce detection + tracking nativamente.
  - YOLO è usato SOLO sul frame anchor di ogni finestra per scegliere
    il prompt migliore da STADIUM_AD_PROMPTS.
  - ByteTrack rimosso completamente.
  - Il video è elaborato a finestre di WINDOW_SIZE frame.
  - Il prompt vincente persiste tra finestre; viene rieletto solo se
    la finestra precedente ha prodotto 0 mask valide.
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import shutil
import torch
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from tqdm import tqdm
from PIL import Image

# SAHI — pip install sahi
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

from sam3.model_builder import build_sam3_image_model, build_sam3_video_predictor
from sam3.model.sam3_image_processor import Sam3Processor

# ── Configurazioni ────────────────────────────────────────────────────────────

# Soglia base per box grandi; per box piccoli viene abbassata automaticamente (fix #2)
YOLO_BOX_CONF_BASE      = 0.40
# Soglia minima assoluta usata da SAHI (molto bassa: il filtro adattivo fa il resto)
YOLO_SAHI_CONF          = 0.10

SAM3_MAX_SIDE           = 768

# Finestra temporale: quanti frame per sessione SAM3 video
WINDOW_SIZE             = 60

# Campionamento YOLO all'interno della finestra per pick_anchor_frame (fix #5)
ANCHOR_SAMPLE_STEP      = 5

# Soglia area (frazione del frame) sotto cui si attiva il crop SAM3 (fix #3)
SMALL_BOX_AREA_THRESH   = 0.02   # 2% del frame
# Padding relativo attorno al ROI nel crop (fix #3)
CROP_PAD_FACTOR         = 0.30

YOLO_MODEL_PATH   = "/content/drive/MyDrive/ProgettoDL/risultati_yolov11nseg/esperimento_seg11_datasetunificato/weights/best.pt"
INPUT_VIDEO_PATH  = "/content/InferenzaJStadium.mp4"
OUTPUT_VIDEO_PATH = "/content/video_output_sam3video_JStadium.mp4"

TEMP_FRAMES_DIR   = "/tmp/sam3_window_frames"

STADIUM_AD_PROMPTS = [
    # ── Termini generici base ─────────────────────────────────────────────────
    "advertising board",
    "stadium ad hoarding",
    "sponsor billboard",
    "commercial advertising panel",
    "pitch-side advertising panel",
    "advertising hoarding",
    "sponsor panel",
    "advertisement board",
    "ad board",
    "sponsor board",

    # ── Posizione perimetrale ─────────────────────────────────────────────────
    "perimeter sponsor hoarding along the field",
    "sideline advertising boards",
    "advertising banners behind the goal",
    "stadium boundary wall with advertisements",
    "field perimeter sponsor signs",
    "pitch perimeter advertising board",
    "touchline advertising hoarding",
    "goal line advertising board",
    "byline sponsor board",
    "behind the goal sponsor hoarding",
    "corner flag advertising panel",
    "pitch boundary advertising fence",
    "along the pitch advertising wall",
    "football field side advertising board",
    "pitch edge sponsor display",

    # ── Forma e geometria ─────────────────────────────────────────────────────
    "long rectangular advertising banner",
    "continuous flat boundary ad board",
    "long horizontal sponsor signs",
    "low rectangular perimeter wall",
    "flat horizontal rectangular panel with logo",
    "wide low banner at pitch level",
    "thin horizontal strip with sponsor name",
    "elongated rectangular sign along the grass",
    "narrow horizontal hoarding at ground level",
    "low flat panel running along the pitch",

    # ── Tecnologia LED / digitale ─────────────────────────────────────────────
    "bright LED advertising screen",
    "digital LED perimeter board",
    "electronic glowing billboard",
    "scrolling digital ad display",
    "animated LED sponsor board",
    "electronic perimeter display screen",
    "digital rotating advertisement panel",
    "glowing electronic sideline screen",
    "LED ribbon board along the pitch",
    "virtual advertising board overlay",
    "high brightness digital signage panel",
    "backlit sponsor display board",

    # ── Stampa statica / tradizionale ─────────────────────────────────────────
    "static printed advertising banner",
    "printed vinyl sponsor hoarding",
    "non-digital flat advertising board",
    "solid color sponsor panel with text",
    "white board with sponsor logo",
    "colored rectangular printed advertisement",

    # ── Contenuto / brand ─────────────────────────────────────────────────────
    "panels displaying brand logos",
    "commercial sponsor graphics along the pitch",
    "colorful advertising text and logos on boards",
    "sponsor name and logo on perimeter board",
    "brand advertisement along the touchline",
    "corporate logo displayed on pitch-side panel",
    "company name on stadium hoarding",
    "product advertisement on football field fence",
    "sponsor livery on perimeter board",
    "branding panel next to the playing field",

    # ── Contesto stadio calcio ────────────────────────────────────────────────
    "football stadium perimeter advertising",
    "soccer match sideline sponsor board",
    "premier league advertising hoarding",
    "champions league pitch-side sponsor panel",
    "football ground commercial display",
    "stadium pitch surroundings advertisement",
    "professional football match ad board",
    "live football game sponsor signage",
    "match day advertising hoarding in stadium",
    "grass-level sponsor board at football game",
    "stadium inner perimeter commercial panel",
    "football club sponsor display fence",
    "rectangular sign at the edge of football pitch",
    "advertising strip running along football field",
    "low wall with sponsor branding at stadium",
]

# ── Helper generici ───────────────────────────────────────────────────────────

def bbox_iou(mask: np.ndarray, box_xyxy: np.ndarray) -> float:
    x1, y1, x2, y2 = box_xyxy.astype(int)
    box_mask = np.zeros_like(mask, dtype=bool)
    box_mask[y1:y2, x1:x2] = True
    intersection = (mask & box_mask).sum()
    union        = (mask | box_mask).sum()
    return intersection / union if union > 0 else 0.0


def resize_image(image_bgr: np.ndarray, max_side: int = 768):
    H, W      = image_bgr.shape[:2]
    long_side = max(H, W)
    if long_side <= max_side:
        return image_bgr, 1.0
    scale   = max_side / long_side
    new_W   = int(W * scale)
    new_H   = int(H * scale)
    resized = cv2.resize(image_bgr, (new_W, new_H), interpolation=cv2.INTER_AREA)
    return resized, scale


def smooth_mask_contour(mask: np.ndarray, epsilon_frac: float = 0.002) -> np.ndarray:
    mask_u8 = mask.astype(np.uint8) * 255
    k       = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, k, iterations=2)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    smooth = np.zeros_like(mask_u8)
    for cnt in contours:
        epsilon = epsilon_frac * cv2.arcLength(cnt, closed=True)
        approx  = cv2.approxPolyDP(cnt, epsilon, closed=True)
        cv2.fillPoly(smooth, [approx], 255)
    return smooth > 127


def annotate_masks_manual(scene: np.ndarray, detections: sv.Detections,
                           opacity: float = 0.6) -> np.ndarray:
    COLORS_BGR = [
        (255,  80,  80), ( 80, 255,  80), ( 80,  80, 255),
        (255, 255,  80), (255,  80, 255), ( 80, 255, 255),
    ]
    output = scene.copy()
    for i, mask in enumerate(detections.mask):
        if mask.sum() == 0:
            continue
        color   = COLORS_BGR[i % len(COLORS_BGR)]
        overlay = scene.copy()
        overlay[mask] = color
        output  = cv2.addWeighted(overlay, opacity, output, 1 - opacity, 0)
        mask_u8     = mask.astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(output, contours, -1, color, thickness=3)
    return output


def free_vram():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

# ── Fix #1 — YOLO con SAHI tiling ────────────────────────────────────────────

def run_yolo_sahi(
    frame_bgr: np.ndarray,
    yolo_model_path: str,
    device: str,
    conf: float = YOLO_SAHI_CONF,
    slice_size: int = 640,
    overlap_ratio: float = 0.25,
) -> sv.Detections:
    """
    Esegue YOLO sul frame intero E su tile sovrapposti (SAHI), poi unisce
    le detection tramite Non-Maximum Merging. Cattura oggetti piccoli/lontani
    che a risoluzione piena verrebbero persi o avrebbero confidenza bassa.

    slice_size   : dimensione (px) di ogni tile — 640 funziona bene per YOLOv11
    overlap_ratio: sovrapposizione tra tile adiacenti (0.25 = 25%)
    conf         : soglia di ingresso molto bassa; il filtro adattivo farà poi
                   il secondo passaggio di pulizia.
    """
    det_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=yolo_model_path,
        confidence_threshold=conf,
        device=device,
    )
    result = get_sliced_prediction(
        image=cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB),
        detection_model=det_model,
        slice_height=slice_size,
        slice_width=slice_size,
        overlap_height_ratio=overlap_ratio,
        overlap_width_ratio=overlap_ratio,
        perform_standard_pred=True,   # predizione anche sul frame intero
        postprocess_type="NMM",       # Non-Maximum Merging: migliore per hoardings lunghi
        postprocess_match_threshold=0.30,
        verbose=0,
    )
    boxes, confs = [], []
    for pred in result.object_prediction_list:
        b = pred.bbox
        boxes.append([b.minx, b.miny, b.maxx, b.maxy])
        confs.append(pred.score.value)

    if not boxes:
        return sv.Detections.empty()

    return sv.Detections(
        xyxy=np.array(boxes, dtype=np.float32),
        confidence=np.array(confs, dtype=np.float32),
        class_id=np.zeros(len(boxes), dtype=int),
    )

# ── Fix #2 — Soglie di confidenza adattive ───────────────────────────────────

def _adaptive_conf_for_box(box_xyxy: np.ndarray, H: int, W: int) -> float:
    """
    Restituisce la soglia di confidenza appropriata per un singolo box.
    Pannelli piccoli (lontani dalla telecamera) producono intrisecamente
    confidenze più basse: abbassare la soglia in modo proporzionale alla loro
    area evita falsi positivi sulle zone grandi.

    area_frac < 0.2%  → soglia 0.12  (pannello molto lontano)
    area_frac < 0.5%  → soglia 0.15
    area_frac < 1%    → soglia 0.20
    area_frac < 3%    → soglia 0.30
    area_frac ≥ 3%    → soglia base  (YOLO_BOX_CONF_BASE)
    """
    area_frac = (
        (box_xyxy[2] - box_xyxy[0]) *
        (box_xyxy[3] - box_xyxy[1])
    ) / (H * W)
    if area_frac < 0.002:
        return 0.12
    elif area_frac < 0.005:
        return 0.15
    elif area_frac < 0.01:
        return 0.20
    elif area_frac < 0.03:
        return 0.30
    return YOLO_BOX_CONF_BASE


def filter_detections_adaptive(
    detections: sv.Detections,
    H: int,
    W: int,
) -> sv.Detections:
    """Filtra le detection applicando la soglia adattiva a ciascun box."""
    if len(detections) == 0:
        return detections
    keep = [
        i for i, box in enumerate(detections.xyxy)
        if detections.confidence[i] >= _adaptive_conf_for_box(box, H, W)
    ]
    if not keep:
        return sv.Detections.empty()
    return detections[np.array(keep)]

# ── Fix #4 — is_adboard_shape adattivo ───────────────────────────────────────

def is_adboard_shape(
    mask: np.ndarray,
    min_aspect: float = 2.5,
    min_area_frac: float = 0.0005,   # era 0.005 — abbassato di un ordine di grandezza
    max_area_frac: float = 0.30,
) -> bool:
    """
    Verifica che la maschera abbia la forma caratteristica di un hoarding.

    Rispetto alla v1:
    - min_area_frac abbassato a 0.0005 per catturare pannelli molto distanti
    - min_aspect abbassato a 2.5 (pannelli di lato / in prospettiva appaiono
      meno allungati)
    - Per maschere molto piccole (area < 0.5%) l'aspect minimo si abbassa
      ulteriormente a 1.8 per compensare la distorsione prospettica.
    """
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return False
    h_img, w_img = mask.shape
    w         = xs.max() - xs.min() + 1
    h         = ys.max() - ys.min() + 1
    if h == 0:
        return False
    aspect    = w / h
    area_frac = mask.sum() / (h_img * w_img)

    # Aspect minimo effettivo: si abbassa per maschere piccole
    effective_min_aspect = min_aspect if area_frac > 0.005 else 1.8

    return (
        aspect >= effective_min_aspect
        and min_area_frac <= area_frac <= max_area_frac
    )

# ── Fix #3 — Selezione prompt con crop + upscale ─────────────────────────────

def _run_prompt_selection(
    image_bgr: np.ndarray,
    yolo_detections: sv.Detections,
    sam_processor: Sam3Processor,
    device: str,
) -> str | None:
    """
    Logica di selezione prompt originale, applicata a un'immagine
    (che può essere il frame intero o un crop).
    """
    image_bgr_sam, scale = resize_image(image_bgr, max_side=SAM3_MAX_SIDE)
    image_pil_sam        = Image.fromarray(cv2.cvtColor(image_bgr_sam, cv2.COLOR_BGR2RGB))
    yolo_found           = len(yolo_detections) > 0

    with torch.inference_mode():
        with torch.autocast(device, dtype=torch.bfloat16):
            shared_state = sam_processor.set_image(image_pil_sam)

    yolo_boxes_sam = yolo_detections.xyxy * scale if yolo_found else None

    if yolo_found:
        high_conf_mask = yolo_detections.confidence >= YOLO_BOX_CONF_BASE
        if high_conf_mask.any():
            hc_boxes      = yolo_boxes_sam[high_conf_mask]
            union_box_sam = np.array([
                hc_boxes[:, 0].min(), hc_boxes[:, 1].min(),
                hc_boxes[:, 2].max(), hc_boxes[:, 3].max(),
            ], dtype=np.float32)
        else:
            # Anche se nessun box supera la soglia base, usa comunque l'union
            # (i box sono stati già filtrati adattativamente)
            union_box_sam = np.array([
                yolo_boxes_sam[:, 0].min(), yolo_boxes_sam[:, 1].min(),
                yolo_boxes_sam[:, 2].max(), yolo_boxes_sam[:, 3].max(),
            ], dtype=np.float32)
    else:
        union_box_sam = None

    best_prompt   = None
    best_combined = -1.0

    for candidate in STADIUM_AD_PROMPTS:
        kwargs = {"state": shared_state, "prompt": candidate}
        if union_box_sam is not None:
            kwargs["boxes"] = torch.tensor(
                union_box_sam, dtype=torch.float32, device=device).unsqueeze(0)
        try:
            with torch.inference_mode():
                with torch.autocast(device, dtype=torch.bfloat16):
                    output = sam_processor.set_text_prompt(**kwargs)
        except TypeError:
            kwargs.pop("boxes", None)
            with torch.inference_mode():
                with torch.autocast(device, dtype=torch.bfloat16):
                    output = sam_processor.set_text_prompt(**kwargs)

        n_masks = output["masks"].shape[0]
        if n_masks == 0:
            del output
            continue

        masks_c  = output["masks"][:, 0].cpu().numpy().astype(bool)
        scores_c = output["scores"].cpu().float().numpy()
        del output

        if yolo_found:
            total_iou = sum(
                max(bbox_iou(masks_c[j], box) for box in yolo_boxes_sam)
                for j in range(n_masks)
            )
            avg_iou   = total_iou / n_masks
            avg_score = scores_c.mean()
            coverage  = min(n_masks / len(yolo_detections), 1.0)
            combined  = avg_iou * 0.5 + avg_score * 0.3 + coverage * 0.2
        else:
            valid_idx = [i for i in range(n_masks) if is_adboard_shape(masks_c[i])]
            if not valid_idx:
                del masks_c, scores_c
                continue
            avg_score = scores_c[valid_idx].mean()
            combined  = avg_score * 0.6 + min(len(valid_idx) / 5.0, 1.0) * 0.4

        if combined > best_combined:
            best_combined = combined
            best_prompt   = candidate
        del masks_c, scores_c

    del shared_state
    free_vram()
    return best_prompt


def select_best_prompt(
    image_bgr: np.ndarray,
    yolo_detections: sv.Detections,
    sam_processor: Sam3Processor,
    device: str,
) -> str | None:
    """
    [fix #3] Se i box YOLO occupano meno di SMALL_BOX_AREA_THRESH del frame,
    ritaglia un'area padded attorno alle detection e ci fa girare SAM3
    a risoluzione effettivamente più alta. Questo simula uno "zoom in"
    sul cluster di pannelli lontani prima della selezione del prompt.

    Se non ci sono detection oppure i box sono grandi, usa il frame intero
    come nella v1.
    """
    H, W = image_bgr.shape[:2]

    use_crop = False
    if len(yolo_detections) > 0:
        areas = (
            (yolo_detections.xyxy[:, 2] - yolo_detections.xyxy[:, 0]) *
            (yolo_detections.xyxy[:, 3] - yolo_detections.xyxy[:, 1])
        ) / (H * W)
        if areas.max() < SMALL_BOX_AREA_THRESH:
            use_crop = True

    if use_crop:
        boxes = yolo_detections.xyxy
        pad_x = int(CROP_PAD_FACTOR * W)
        pad_y = int(CROP_PAD_FACTOR * H)
        cx1   = int(max(0,   boxes[:, 0].min() - pad_x))
        cy1   = int(max(0,   boxes[:, 1].min() - pad_y))
        cx2   = int(min(W,   boxes[:, 2].max() + pad_x))
        cy2   = int(min(H,   boxes[:, 3].max() + pad_y))

        crop_bgr = image_bgr[cy1:cy2, cx1:cx2]

        # Trasla le coordinate dei box nello spazio del crop
        offset      = np.array([cx1, cy1, cx1, cy1], dtype=np.float32)
        crop_xyxy   = np.clip(yolo_detections.xyxy - offset, 0, None)
        crop_det    = sv.Detections(
            xyxy=crop_xyxy,
            confidence=yolo_detections.confidence,
            class_id=yolo_detections.class_id,
        )
        return _run_prompt_selection(crop_bgr, crop_det, sam_processor, device)

    return _run_prompt_selection(image_bgr, yolo_detections, sam_processor, device)

# ── Fix #5 — Anchor frame smart ──────────────────────────────────────────────

def pick_anchor_frame(
    window_frames: list[np.ndarray],
    yolo_model_path: str,
    device: str,
    sample_step: int = ANCHOR_SAMPLE_STEP,
    H: int = 0,
    W: int = 0,
) -> tuple[int, np.ndarray, sv.Detections]:
    """
    Campiona YOLO ogni `sample_step` frame nella finestra e restituisce
    il frame con il maggior numero di detection (dopo il filtro adattivo).
    Questo assicura che il prompt venga selezionato sulla vista più
    informativa della finestra, non necessariamente il frame 0.

    Restituisce: (anchor_idx, anchor_frame, anchor_detections)
    """
    best_idx  = 0
    best_n    = -1
    best_dets = sv.Detections.empty()

    for i in range(0, len(window_frames), sample_step):
        frame = window_frames[i]
        dets  = run_yolo_sahi(frame, yolo_model_path, device)
        dets  = filter_detections_adaptive(dets, H or frame.shape[0], W or frame.shape[1])

        n = len(dets)
        if n > best_n:
            best_n    = n
            best_idx  = i
            best_dets = dets

    # Se nessun frame campionato ha detection, ricade sul frame 0
    if best_n <= 0:
        frame0 = window_frames[0]
        dets0  = run_yolo_sahi(frame0, yolo_model_path, device)
        dets0  = filter_detections_adaptive(dets0, H or frame0.shape[0], W or frame0.shape[1])
        return 0, frame0, dets0

    return best_idx, window_frames[best_idx], best_dets

# ── Gestione finestre JPEG ────────────────────────────────────────────────────

def dump_window_to_jpeg(frames: list[np.ndarray], out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    for i, frame in enumerate(frames):
        path = os.path.join(out_dir, f"{i:05d}.jpg")
        cv2.imwrite(path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

# ── Conversione output SAM3 video → maschere numpy @risoluzione originale ────

def parse_video_output(
    outputs: dict,
    H_orig: int,
    W_orig: int,
) -> tuple[np.ndarray | None, np.ndarray | None, list[float]]:
    """
    Converte l'output di un frame SAM3 video in:
      masks_orig : bool ndarray [N, H_orig, W_orig]
      boxes_orig : float ndarray [N, 4] in pixel xyxy
      scores     : list[float]
    """
    masks_raw  = outputs.get("out_binary_masks")
    boxes_xywh = outputs.get("out_boxes_xywh")
    probs      = outputs.get("out_probs", [])

    if masks_raw is None or len(masks_raw) == 0:
        return None, None, []

    N = len(masks_raw)

    masks_orig = np.stack([
        cv2.resize(
            m.astype(np.uint8) * 255,
            (W_orig, H_orig),
            interpolation=cv2.INTER_NEAREST,
        ) > 127
        for m in masks_raw
    ])  # [N, H_orig, W_orig]

    boxes_orig = np.zeros((N, 4), dtype=np.float32)
    for i, (cx, cy, bw, bh) in enumerate(boxes_xywh):
        boxes_orig[i] = [
            cx * W_orig,
            cy * H_orig,
            (cx + bw) * W_orig,
            (cy + bh) * H_orig,
        ]

    scores = list(probs) if len(probs) == N else [1.0] * N
    return masks_orig, boxes_orig, scores

# ── Annotazione frame ─────────────────────────────────────────────────────────

def annotate_frame(
    image_bgr: np.ndarray,
    masks_orig: np.ndarray | None,
    boxes_orig: np.ndarray | None,
    scores: list[float],
) -> np.ndarray:
    annotated = image_bgr.copy()
    if masks_orig is None or len(masks_orig) == 0:
        return annotated

    final_masks, final_scores, final_boxes = [], [], []

    for i, m in enumerate(masks_orig):
        smoothed = smooth_mask_contour(m)
        ys, xs   = np.where(smoothed)
        if len(xs) == 0:
            continue
        if not is_adboard_shape(smoothed):
            continue
        box = np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=float)
        final_masks.append(smoothed)
        final_boxes.append(box)
        final_scores.append(scores[i] if i < len(scores) else 1.0)

    if not final_masks:
        return annotated

    masks_np  = np.stack(final_masks)
    boxes_np  = np.stack(final_boxes)
    scores_np = np.array(final_scores, dtype=np.float32)

    sam_det = sv.Detections(
        xyxy=boxes_np, mask=masks_np, confidence=scores_np,
        class_id=np.zeros(len(boxes_np), dtype=int),
    )

    annotated = annotate_masks_manual(annotated, sam_det, opacity=0.4)
    annotated = sv.BoxAnnotator(
        thickness=2,
        color_lookup=sv.ColorLookup.INDEX,
    ).annotate(scene=annotated, detections=sam_det)
    annotated = sv.LabelAnnotator(
        text_scale=0.5, text_thickness=1,
        color_lookup=sv.ColorLookup.INDEX,
    ).annotate(
        scene=annotated, detections=sam_det,
        labels=[f"adboard {s:.2f}" for s in scores_np],
    )

    return annotated

# ── Pipeline principale ───────────────────────────────────────────────────────

def process_video():
    if not os.path.exists(YOLO_MODEL_PATH):
        raise FileNotFoundError(f"Modello YOLO non trovato: {YOLO_MODEL_PATH}")
    if not os.path.exists(INPUT_VIDEO_PATH):
        raise FileNotFoundError(f"Video non trovato: {INPUT_VIDEO_PATH}")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    device = "cuda" if torch.cuda.is_available() else "cpu"

    free_vram()

    # YOLO non viene caricato come modello standalone: viene istanziato
    # internamente da SAHI a ogni chiamata run_yolo_sahi. Questo permette
    # a SAHI di gestirne il ciclo di vita e il device placement.

    print("Caricamento SAM3 image model (per selezione prompt)...")
    sam_image_model = build_sam3_image_model()
    sam_processor   = Sam3Processor(sam_image_model, confidence_threshold=0.3)
    sam_image_model.to(device)

    print("Caricamento SAM3 video predictor...")
    video_predictor = build_sam3_video_predictor()

    # Lettura video
    cap          = cv2.VideoCapture(INPUT_VIDEO_PATH)
    fps          = int(cap.get(cv2.CAP_PROP_FPS))
    W_orig       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H_orig       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (W_orig, H_orig))

    best_prompt: str | None = None  # persiste tra finestre

    pbar = tqdm(total=total_frames, desc="Elaborazione Frame")

    window_start    = 0
    all_frames_done = False

    while not all_frames_done:
        # ── Leggi i frame della finestra corrente ──────────────────────────
        window_frames: list[np.ndarray] = []
        for _ in range(WINDOW_SIZE):
            ret, frame = cap.read()
            if not ret:
                all_frames_done = True
                break
            window_frames.append(frame)

        if not window_frames:
            break

        n_window = len(window_frames)

        # ── [fix #5] Scelta anchor frame smart ────────────────────────────
        print(f"\n[finestra {window_start}] Campionamento YOLO per anchor frame...")
        anchor_idx, anchor, yolo_detections = pick_anchor_frame(
            window_frames,
            YOLO_MODEL_PATH,
            device,
            sample_step=ANCHOR_SAMPLE_STEP,
            H=H_orig,
            W=W_orig,
        )
        print(f"  → anchor: frame locale {anchor_idx}, "
              f"{len(yolo_detections)} detection")

        # ── Selezione prompt (se serve) ────────────────────────────────────
        need_reelect = best_prompt is None
        if need_reelect:
            print(f"[finestra {window_start}] Selezione prompt...")
            best_prompt = select_best_prompt(
                anchor, yolo_detections, sam_processor, device)

            if best_prompt:
                print(f"  → prompt scelto: '{best_prompt}'")
            else:
                print("  → nessun prompt valido trovato, skip finestra")
                for f in window_frames:
                    out.write(f)
                pbar.update(n_window)
                window_start += n_window
                continue

        # ── Scrivi i frame come JPEG per la sessione SAM3 video ───────────
        if os.path.exists(TEMP_FRAMES_DIR):
            shutil.rmtree(TEMP_FRAMES_DIR)
        dump_window_to_jpeg(window_frames, TEMP_FRAMES_DIR)

        # ── Apri sessione SAM3 video e propaga ────────────────────────────
        session_id = None
        try:
            response   = video_predictor.handle_request(
                request=dict(type="start_session", resource_path=TEMP_FRAMES_DIR))
            session_id = response["session_id"]

            # Usa l'anchor scelto da fix #5 come frame di partenza del prompt
            prompt_response = video_predictor.handle_request(
                request=dict(
                    type="add_prompt",
                    session_id=session_id,
                    frame_index=anchor_idx,
                    text=best_prompt,
                )
            )

            anchor_outputs  = prompt_response["outputs"]
            found_at_anchor = (
                anchor_outputs is not None
                and len(anchor_outputs.get("out_binary_masks", [])) > 0
            )

            if not found_at_anchor:
                # Prompt non funziona su questo anchor: rieleziona
                print(f"[finestra {window_start}] Prompt '{best_prompt}' "
                      f"senza detection su anchor {anchor_idx}, rielezione...")
                video_predictor.handle_request(
                    request=dict(type="close_session", session_id=session_id))
                session_id  = None
                best_prompt = select_best_prompt(
                    anchor, yolo_detections, sam_processor, device)

                if best_prompt is None:
                    for f in window_frames:
                        out.write(f)
                    pbar.update(n_window)
                    window_start += n_window
                    continue

                # Riapri sessione con il nuovo prompt
                response    = video_predictor.handle_request(
                    request=dict(type="start_session", resource_path=TEMP_FRAMES_DIR))
                session_id  = response["session_id"]
                prompt_response = video_predictor.handle_request(
                    request=dict(
                        type="add_prompt",
                        session_id=session_id,
                        frame_index=anchor_idx,
                        text=best_prompt,
                    )
                )
                anchor_outputs = prompt_response["outputs"]

            # ── Propagazione bidirezionale dalla posizione dell'anchor ─────
            # Propagazione forward (frame anchor → fine finestra)
            outputs_per_frame: dict[int, dict] = {}
            outputs_per_frame[anchor_idx] = anchor_outputs

            for frame_response in video_predictor.handle_stream_request(
                request=dict(
                    type="propagate_in_video",
                    session_id=session_id,
                    direction="forward",
                )
            ):
                fidx = frame_response["frame_index"]
                outputs_per_frame[fidx] = frame_response["outputs"]

            # Propagazione backward (frame anchor → inizio finestra)
            # Solo se l'anchor non è il frame 0
            if anchor_idx > 0:
                for frame_response in video_predictor.handle_stream_request(
                    request=dict(
                        type="propagate_in_video",
                        session_id=session_id,
                        direction="backward",
                    )
                ):
                    fidx = frame_response["frame_index"]
                    if fidx not in outputs_per_frame:
                        outputs_per_frame[fidx] = frame_response["outputs"]

        finally:
            if session_id is not None:
                try:
                    video_predictor.handle_request(
                        request=dict(type="close_session", session_id=session_id))
                except Exception:
                    pass
            free_vram()

        # ── Annota e scrivi ogni frame della finestra ──────────────────────
        produced_any_mask = False
        for local_idx, frame_bgr in enumerate(window_frames):
            frame_outputs = outputs_per_frame.get(local_idx)

            if frame_outputs is not None:
                masks_orig, boxes_orig, scores = parse_video_output(
                    frame_outputs, H_orig, W_orig)
            else:
                masks_orig, boxes_orig, scores = None, None, []

            if masks_orig is not None and len(masks_orig) > 0:
                produced_any_mask = True

            annotated = annotate_frame(frame_bgr, masks_orig, boxes_orig, scores)
            out.write(annotated)
            pbar.update(1)

        # Se nessun frame ha prodotto mask, forza rielezione alla prossima finestra
        if not produced_any_mask:
            print(f"[finestra {window_start}] Nessuna mask prodotta, "
                  f"rielezione forzata alla prossima finestra.")
            best_prompt = None

        window_start += n_window

    cap.release()
    out.release()
    pbar.close()

    # Pulizia
    if os.path.exists(TEMP_FRAMES_DIR):
        shutil.rmtree(TEMP_FRAMES_DIR)

    del sam_processor, sam_image_model, video_predictor
    free_vram()
    print(f"\nElaborazione conclusa. Salvato in: {OUTPUT_VIDEO_PATH}")


if __name__ == "__main__":
    process_video()

Caricamento SAM3 image model (per selezione prompt)...


config.json: 0.00B [00:00, ?B/s]

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

INFO 2026-04-12 23:44:57,776 6719 sam3_video_predictor.py: 109: using the following GPU IDs: [0]
INFO 2026-04-12 23:44:57,777 6719 sam3_video_predictor.py: 125: 


	*** START loading model on all ranks ***


INFO 2026-04-12 23:44:57,777 6719 sam3_video_predictor.py: 127: loading model on rank=0 with world_size=1 -- this could take a while ...


Caricamento SAM3 video predictor...


...

INFO 2026-04-13 00:35:03,809 6719 sam3_base_predictor.py: 286: propagation ended in session 9024097e-031a-439e-901d-100907333a05


  0%|          | 0/46 [00:00<?, ?it/s]

0it [00:00, ?it/s]

INFO 2026-04-13 00:35:04,724 6719 sam3_base_predictor.py: 286: propagation ended in session 9024097e-031a-439e-901d-100907333a05
INFO 2026-04-13 00:35:04,995 6719 sam3_base_predictor.py: 305: removed session 9024097e-031a-439e-901d-100907333a05
Elaborazione Frame: 100%|██████████| 3286/3286 [50:11<00:00,  1.09it/s]



Elaborazione conclusa. Salvato in: /content/video_output_sam3video_JStadium.mp4
